In [80]:
import glob
import pandas as pd
import os

import glob
import shutil
import re
import unicodedata
import hashlib
from textgrids import TextGrid

In [ ]:
def make_dataset_df(lang, root = '.'):
    all_files = glob.glob(f'{root}/{lang}/*/**/***/****/*****')
    data = [file.split(f'{root}/{lang}/')[1].split('/') for file in all_files]

    df = pd.DataFrame(data = data, columns=['singer', 'style', 'song', 'group', 'file'])
    df['input_path'] = all_files

    # prepare output paths
    if lang in ['ZH', 'KO', 'JA', 'RU']:
        df['song'] = df['song'].apply(normalize_hash)
    else:
        df['song'] = df['song'].apply(normalize)

    df['new_song'] = df['style'] + '_' +  df['song'] + '_' + df['group']
    df['out_path'] = df['singer'] + '-' + df['new_song'] + '-' + df['file'].str.replace('_TextGrid', '.TextGrid')

    return df

def make_wav_tg_dfs(df, 
                    select_styles = ['Vibrato', 'Glissando'],
                    output_folder = '.'):

    EXCLUDE_GROUPS = ['Paired_Speech_Group', 'Control_Group']

    df_songs = df[~df['group'].isin(EXCLUDE_GROUPS)]

    df_songs = df_songs[df_songs['style'].isin(select_styles)]

    df_wav = df_songs[df_songs['file'].str.contains('wav')].sort_values('input_path')
    df_wav['out_path'] = f'{output_folder}/wav/' + df_wav['out_path']

    df_tg = df_songs[df_songs['file'].str.contains('TextGrid')].sort_values('input_path')
    df_tg['out_path'] = f'{output_folder}/TextGrid/' + df_tg['out_path']
    df_tg['duration'] = df_tg['input_path'].apply(get_tg_length)

    total_duration = df_tg['duration'].sum()/60.
    number_null_song_names = len(df_wav[df_wav['song'].isna()])

    if len(df_wav) == len(df_tg):
        print(f'Songs in dataset: {len(df_wav)}')
    else:
        print('Not same number of wavs and TextGrids!')

    print(f'Total duration of the dataset: {int(total_duration)} min')
    print(f'Songs with null names: {number_null_song_names}')
    
    return df_wav, df_tg

############################################################

def normalize_hash(text):
    return normalize(text, enc_hash = True)

def normalize(text, enc_hash = False):
    if pd.isnull(text):
        return ""
    # Lowercase
    text = text.lower()
    # Remove accents (e.g., é → e)
    if enc_hash:
        #text = unicodedata.normalize('NFKD', text)
        text = hashlib.sha1(text.encode("utf-8")).hexdigest()[:8]
    else:
        text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text)
    # Strip leading/trailing spaces
    return text.strip().replace(' ', '_')

############################################################

def check_matching_files(output_folder):
    tg_root = f'{output_folder}/TextGrid'
    wav_root = f'{output_folder}/wav'

    tg_files = sorted(glob.glob(f'{tg_root}/*'))
    wav_files = sorted(glob.glob(f'{wav_root}/*'))

    tg_stems = [file.split('/')[-1].split('.TextGrid')[0] for file in tg_files]
    wav_stems = [file.split('/')[-1].split('.wav')[0] for file in wav_files]

    df_tg = pd.DataFrame(data = tg_stems)
    df_wav = pd.DataFrame(data = wav_stems)

    n_wav = len(df_wav)
    n_tg = len(df_tg)

    matches = (df_tg == df_wav).sum().values[0]

    return n_wav, n_tg, matches

def get_all_tg_files(output_folder):
    tg_root = f'{output_folder}/TextGrid'
    tg_files = sorted(glob.glob(f'{tg_root}/*'))

    return tg_files

def get_total_length(lang):
    files = get_all_tg_files(lang)
    secs = sum([get_tg_length(f) for f in files])
    return secs/60.

def get_tg_length(file):
    tg = TextGrid()
    tg.read(file)

    return tg['global'].xmax

In [110]:
ROOT = '/Users/tomasandrade/Documents/BSC/ICHOIR/datasets/GTSinger'

lang = 'JA'

OUTPUT_FOLDER = f'{ROOT}/GTSinger_{lang}'

os.makedirs(f'{OUTPUT_FOLDER}/wav')
os.makedirs(f'{OUTPUT_FOLDER}/TextGrid')

In [135]:
df = make_dataset_df(lang, root = ROOT)

In [136]:
df['style'].unique()

array(['Vibrato', 'Pharyngeal', 'Breathy', 'Glissando',
       'Mixed_Voice_and_Falsetto'], dtype=object)

In [ ]:
df_wav, df_tg = make_wav_tg_dfs(df, 
                            select_styles = ['Vibrato', 'Glissando'],
                            output_folder = OUTPUT_FOLDER)

Songs in dataset: 240
Total duration of the dataset: 46 min
Songs with null names: 0


In [ ]:
for input_path, out_path in zip(df_wav['input_path'], df_wav['out_path']):
    shutil.copy2(input_path, out_path)

for input_path, out_path in zip(df_tg['input_path'], df_tg['out_path']):
    shutil.copy2(input_path, out_path)

In [107]:
check_matching_files(OUTPUT_FOLDER), get_total_length(OUTPUT_FOLDER)

((355, 355, 355), 69.19329999999995)